# In-domain training on OOD features CSV (from Drive)

Loads a **precomputed features CSV** from Google Drive, **stratified train/test split**, then runs the same style of pipeline as **Step 7**: `SelectKBest` × `StandardScaler` × multinomial **LogisticRegression** over a **C** grid, separately for **STATIC**, **PROMPT**, and **COMBO**.

You need:
- Your merged features file (e.g. `ood/onestopenglish_features_mistral-7b.csv`).
- `feature_schema_<tag>.json` from Step 7 artifacts (column lists).

This does **not** modify Step 7; it only trains **on the OOD corpus** and reports **test macro-F1**.

In [ ]:
# --- Edit these ---
BASE_DIR = '/content/drive/MyDrive/BeyondFK'  # Colab Drive root for your project

# Relative to BASE_DIR: your merged features CSV
FEATURES_CSV_REL = 'ood/onestopenglish_features_mistral-7b.csv'

# Step 7 artifact schema (tells us static / prompt / combo column names)
MODEL_TAG = 'mistral-7b'
SCHEMA_JSON_REL = f'results/by_prompt_model/{MODEL_TAG}/artifacts/feature_schema_{MODEL_TAG}.json'

TEST_SIZE = 0.2
RANDOM_STATE = 42
C_GRID = [0.1, 0.3, 1.0, 3.0, 10.0]  # same default as Step 7

# Where to save JSON summary (under BASE_DIR)
OUT_JSON_REL = f'ood/results/onestopenglish_in_domain_train_{MODEL_TAG}.json'

print('FEATURES_CSV:', FEATURES_CSV_REL)
print('SCHEMA_JSON :', SCHEMA_JSON_REL)


In [ ]:
import os
import re
import json

import numpy as np
import pandas as pd
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass  # local: set BASE_DIR in previous cell to '.'

FEATURES_CSV = os.path.join(BASE_DIR, FEATURES_CSV_REL)
SCHEMA_JSON = os.path.join(BASE_DIR, SCHEMA_JSON_REL)
OUT_JSON = os.path.join(BASE_DIR, OUT_JSON_REL)
os.makedirs(os.path.dirname(OUT_JSON), exist_ok=True)

print('FEATURES_CSV abs:', os.path.abspath(FEATURES_CSV))
print('SCHEMA_JSON  abs:', os.path.abspath(SCHEMA_JSON))


In [ ]:
def normalize_prompt_headers(df):
    rename_map = {}
    for c in df.columns:
        m = re.match(r'^prompt_(\d+)$', str(c).strip())
        if m:
            rename_map[c] = f"prompt_{int(m.group(1))}"
    return df.rename(columns=rename_map)


def prep_xy(df, feature_cols, label_col):
    X = df[feature_cols].copy()
    X = X.apply(pd.to_numeric, errors='coerce').fillna(0).replace([np.inf, -np.inf], 0)
    y = df[label_col].astype(str).values
    return X, y


def k_grid_static_prompt(n_feat):
    if n_feat <= 0:
        return []
    if n_feat < 5:
        return [n_feat]
    ks = list(range(5, n_feat + 1, 5))
    if n_feat not in ks:
        ks.append(n_feat)
    return sorted(set(ks))


def k_grid_combo(n_feat):
    if n_feat <= 0:
        return []
    if n_feat < 10:
        return [n_feat]
    ks = sorted(set(range(10, n_feat + 1, 10)) | {n_feat})
    return [min(k, n_feat) for k in ks]


def grid_search(name, X_train, X_test, y_train, y_test, k_list, c_grid):
    le = LabelEncoder()
    y_tr = le.fit_transform(y_train)
    y_te = le.transform(y_test)
    best = {'f1': -1.0, 'k': None, 'C': None, 'selector': None, 'scaler': None}
    n_feat = X_train.shape[1]
    for k in k_list:
        kk = min(int(k), n_feat)
        if kk < 1:
            continue
        selector = SelectKBest(f_classif, k=kk)
        Xtr = selector.fit_transform(X_train, y_tr)
        Xte = selector.transform(X_test)
        scaler = StandardScaler()
        Xtr = scaler.fit_transform(Xtr)
        Xte = scaler.transform(Xte)
        for C in c_grid:
            clf = LogisticRegression(
                multi_class='multinomial', solver='lbfgs', max_iter=20000,
                random_state=42, C=C,
            )
            clf.fit(Xtr, y_tr)
            pred_te = clf.predict(Xte)
            f1 = f1_score(y_te, pred_te, average='macro')
            if f1 > best['f1']:
                best.update({'f1': f1, 'k': kk, 'C': C, 'selector': selector, 'scaler': scaler})
    if best['selector'] is None:
        return {'name': name, 'macro_f1': None, 'error': 'no valid (k, C) grid point'}
    selector, scaler, C = best['selector'], best['scaler'], best['C']
    Xtr_sel = scaler.transform(selector.transform(X_train))
    Xte_sel = scaler.transform(selector.transform(X_test))
    clf_final = LogisticRegression(
        multi_class='multinomial', solver='lbfgs', max_iter=20000,
        random_state=42, C=C,
    )
    clf_final.fit(Xtr_sel, y_tr)
    pred = clf_final.predict(Xte_sel)
    pred_labels = le.inverse_transform(pred)
    macro = float(f1_score(y_test, pred_labels, average='macro'))
    report = classification_report(y_test, pred_labels, output_dict=True, zero_division=0)
    return {
        'name': name,
        'best_k': int(best['k']),
        'best_C': float(best['C']),
        'macro_f1': macro,
        'classification_report': report,
    }

print('Helpers defined.')


In [ ]:
assert os.path.isfile(FEATURES_CSV), f'Missing CSV: {FEATURES_CSV}'
assert os.path.isfile(SCHEMA_JSON), f'Missing schema: {SCHEMA_JSON}'

with open(SCHEMA_JSON, 'r', encoding='utf-8') as f:
    schema = json.load(f)

label_col = schema.get('label_col', 'education_level')
static_cols = list(schema.get('static_columns', []))
prompt_cols = list(schema.get('prompt_columns', []))
combo_cols = list(schema.get('combo_columns', []))

df = pd.read_csv(FEATURES_CSV)
df.columns = df.columns.astype(str).str.strip()
df = normalize_prompt_headers(df)
df = df.dropna(subset=[label_col])

y_raw = df[label_col].astype(str)
if y_raw.value_counts().min() < 2:
    train_df, test_df = train_test_split(
        df, test_size=TEST_SIZE, random_state=RANDOM_STATE, shuffle=True
    )
else:
    train_df, test_df = train_test_split(
        df, test_size=TEST_SIZE, random_state=RANDOM_STATE, shuffle=True, stratify=y_raw
    )

def cols_present(cols):
    return [c for c in cols if c in train_df.columns]

static_use = cols_present(static_cols)
prompt_use = cols_present(prompt_cols)
combo_use = cols_present(combo_cols)

print('Rows:', len(df), '| Train:', len(train_df), '| Test:', len(test_df))
print('Static cols in CSV:', len(static_use), '| Prompt:', len(prompt_use), '| Combo:', len(combo_use))


In [ ]:
results = {}

if len(static_use) < 1:
    results['static'] = {'error': 'no static columns'}
else:
    X_tr, y_tr = prep_xy(train_df, static_use, label_col)
    X_te, y_te = prep_xy(test_df, static_use, label_col)
    results['static'] = grid_search(
        'STATIC', X_tr, X_te, y_tr, y_te, k_grid_static_prompt(X_tr.shape[1]), C_GRID
    )

if len(prompt_use) < 1:
    results['prompt'] = {'error': 'no prompt columns'}
else:
    X_tr, y_tr = prep_xy(train_df, prompt_use, label_col)
    X_te, y_te = prep_xy(test_df, prompt_use, label_col)
    results['prompt'] = grid_search(
        'PROMPT', X_tr, X_te, y_tr, y_te, k_grid_static_prompt(X_tr.shape[1]), C_GRID
    )

if len(combo_use) < 1:
    results['combo'] = {'error': 'no combo columns'}
else:
    X_tr, y_tr = prep_xy(train_df, combo_use, label_col)
    X_te, y_te = prep_xy(test_df, combo_use, label_col)
    results['combo'] = grid_search(
        'COMBO', X_tr, X_te, y_tr, y_te, k_grid_combo(X_tr.shape[1]), C_GRID
    )

for key in ('static', 'prompt', 'combo'):
    r = results[key]
    if r.get('macro_f1') is not None:
        print(f"{key.upper():6s} test macro-F1={r['macro_f1']:.4f}  best_k={r['best_k']}  best_C={r['best_C']}")
    else:
        print(key.upper(), r)


In [ ]:
rows = []
for key in ('static', 'prompt', 'combo'):
    r = results.get(key, {})
    if r.get('macro_f1') is None:
        continue
    rep = r['classification_report']
    rows.append({
        'setting': key.upper(),
        'macro_f1': r['macro_f1'],
        'best_k': r['best_k'],
        'best_C': r['best_C'],
        'f1_elementary': rep.get('elementary', {}).get('f1-score'),
        'f1_middle': rep.get('middle', {}).get('f1-score'),
        'f1_high': rep.get('high', {}).get('f1-score'),
        'accuracy': rep.get('accuracy'),
    })

summary = pd.DataFrame(rows)
display(summary)

payload = {
    'features_csv': FEATURES_CSV,
    'schema_json': SCHEMA_JSON,
    'label_col': label_col,
    'n_total': int(len(df)),
    'n_train': int(len(train_df)),
    'n_test': int(len(test_df)),
    'test_size': TEST_SIZE,
    'random_state': RANDOM_STATE,
    'c_grid': C_GRID,
    'results': results,
}

with open(OUT_JSON, 'w', encoding='utf-8') as f:
    json.dump(payload, f, indent=2)
print('Saved:', OUT_JSON)
